# Notebook 04a: Grad-CAM Explainability

Generate Grad-CAM, Grad-CAM++, and Score-CAM heatmaps to verify that the Modified ConvNeXt attends to clinically meaningful regions.

Requires: `pip install grad-cam`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

from src.models.builder import build_model
from src.utils.explainability import make_eval_transform, get_target_layer

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

## Load trained model

In [ ]:
WEIGHTS = '../runs/cv_modified_convnext/fold0/best.pt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = build_model('modified_convnext', num_classes=4, pretrained=False).to(device)
state = torch.load(WEIGHTS, map_location='cpu')
if 'model' in state: state = state['model']
model.load_state_dict(state, strict=False)
model.eval()
print('Model loaded')

## Generate heatmaps

In [ ]:
CLASS_NAMES = ['Adeno', 'SCC', 'SCLC', 'Normal']
SAMPLE_IMAGES = sorted(Path('../data/sample_test_images').glob('*.jpg'))[:4]

target_layers = get_target_layer(model)
tf = make_eval_transform(image_size=256)
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

fig, axes = plt.subplots(2, len(SAMPLE_IMAGES), figsize=(13, 6.5))
for i, img_path in enumerate(SAMPLE_IMAGES):
    pil = Image.open(img_path).convert('RGB').resize((256, 256))
    rgb = np.array(pil) / 255.0
    x = tf(pil).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())
    grayscale = cam(input_tensor=x, targets=None)[0]
    viz = show_cam_on_image(rgb, grayscale, use_rgb=True)

    axes[0, i].imshow(rgb); axes[0, i].axis('off')
    axes[0, i].set_title(f'Input ({img_path.stem})', fontsize=10)
    axes[1, i].imshow(viz); axes[1, i].axis('off')
    axes[1, i].set_title(f'Pred: {CLASS_NAMES[pred]} (p={probs[pred]:.3f})', fontsize=10)
plt.tight_layout(); plt.show()

## Save heatmaps to disk

In [ ]:
from src.utils.explainability import generate_heatmaps
OUT_DIR = Path('../runs/gradcam_outputs')
results = generate_heatmaps(model, SAMPLE_IMAGES, OUT_DIR, device=device)
print(f'Saved {len(results)} heatmaps to {OUT_DIR}')